# 06 - Cold-Start Routing

Demonstrates `SpotifyColdStartRouter` (`src/models/fallback_handler.py`),
which decides *how* to score a candidate track for a playlist depending on
how much data is available:

1. **Playlist cold-start** (few tracks in the playlist so far) -> fall back
   to global popularity.
2. **Track cold-start** (track has few interactions across all playlists)
   -> score via content similarity to the tracks already in the playlist.
3. **Warm playlist + warm track** -> score via the trained LightFM
   collaborative model.

This notebook also documents a real bug we found and fixed in this file:
the original `_compute_content_score` accepted a `playlist_idx` argument
but never used it - it returned a track's average similarity to the
*entire catalog*, which has nothing to do with the specific playlist being
scored. It's fixed to compare a candidate track against the tracks already
in the target playlist instead.

In [1]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import pickle
from scipy import sparse
from src.models.fallback_handler import SpotifyColdStartRouter

tracks = pd.read_csv('../data/processed/tracks_clean.csv')
tracks['track_uri'] = 'spotify:track:' + tracks['track_id'].astype(str)
interactions_df = pd.read_csv('../data/processed/playlist_track_interactions.csv')

with open('../data/processed/id_mappings.pkl', 'rb') as f:
    mappings = pickle.load(f)
track_to_idx = mappings['track_to_idx']

with open('../data/processed/lightfm_model.pkl', 'rb') as f:
    hybrid_model = pickle.load(f)

content = np.load('../data/processed/content_embeddings.npz', allow_pickle=True)
emb_uris = list(content['track_uris'])
emb_matrix = content['embeddings']
uri_to_emb_idx = {u: i for i, u in enumerate(emb_uris)}

print("Tracks:", len(tracks), " Playlists:", interactions_df['pid'].nunique())

Tracks: 16804  Playlists: 674


## Build the candidate pool and per-track interaction counts

In [2]:
interaction_counts = interactions_df.groupby('track_uri').size().rename('interaction_count')

candidate_df = tracks[['track_uri', 'track_name', 'artists', 'album_name', 'popularity']].copy()
candidate_df = candidate_df.rename(columns={'artists': 'artist_name'})
candidate_df = candidate_df[candidate_df['track_uri'].isin(track_to_idx)].reset_index(drop=True)
candidate_df['track_idx'] = candidate_df['track_uri'].map(track_to_idx)
candidate_df['interaction_count'] = candidate_df['track_uri'].map(interaction_counts).fillna(0).astype(int)
candidate_df = candidate_df[candidate_df['track_uri'].isin(uri_to_emb_idx)].reset_index(drop=True)

print(candidate_df.shape)
candidate_df.head()

(16804, 7)


,track_uri,track_name,artist_name,album_name,popularity,track_idx,interaction_count
0,spotify:track:3nqQXoyQOWXiESFLlDF1hG,Unholy (feat. Kim Petras),Sam Smith;Kim Petras,Unholy (feat. Kim Petras),100,8300,16
1,spotify:track:2tTmW7RDtMQtBk7m2rYeSw,"Quevedo: Bzrp Music Sessions, Vol. 52",Bizarrap;Quevedo,"Quevedo: Bzrp Music Sessions, Vol. 52",99,6317,16
2,spotify:track:5ww2BF9slyYgNOk37BlC4u,La Bachata,Manuel Turizo,La Bachata,98,12884,16
3,spotify:track:4uUG5RXrOk84mYEfFvj3cK,I'm Good (Blue),David Guetta;Bebe Rexha,I'm Good (Blue),98,10630,16
4,spotify:track:1IHWl5LamUGEuP4ozKQSXZ,Tití Me Preguntó,Bad Bunny,Un Verano Sin Ti,97,2815,16


## Build a content similarity matrix restricted to our candidate pool

(Full catalog x catalog similarity would be a ~17k x 17k dense matrix - fine at this scale, but we build it once and reuse it.)

In [3]:
cand_emb_idx = candidate_df['track_uri'].map(uri_to_emb_idx).values
cand_embeddings = emb_matrix[cand_emb_idx]
sim_matrix_full = cand_embeddings @ cand_embeddings.T
print("Similarity matrix shape:", sim_matrix_full.shape)

# re-index candidate_df's track_idx to positions within this local similarity matrix
local_idx_map = {uri: i for i, uri in enumerate(candidate_df['track_uri'])}
candidate_df['local_idx'] = candidate_df['track_uri'].map(local_idx_map)

Similarity matrix shape: (16804, 16804)


## Case 1: Playlist cold-start

A brand-new playlist with only 1 track -> should fall back to popularity,
regardless of what the collaborative or content models think.

In [4]:
router = SpotifyColdStartRouter(warm_playlist_threshold=5, warm_track_threshold=10)

recs_cold_playlist = router.route_recommendations(
    playlist_idx=0,
    playlist_track_count=1,   # below warm_playlist_threshold
    candidate_tracks=candidate_df.assign(track_idx=candidate_df['local_idx']),
    hybrid_model=hybrid_model,
    content_similarity_matrix=sim_matrix_full,
)
candidate_df.set_index('track_uri').loc[recs_cold_playlist, ['track_name', 'artist_name', 'popularity']]

,track_name,artist_name,popularity
track_uri,,,
spotify:track:3nqQXoyQOWXiESFLlDF1hG,Unholy (feat. Kim Petras),Sam Smith;Kim Petras,100
spotify:track:2tTmW7RDtMQtBk7m2rYeSw,"Quevedo: Bzrp Music Sessions, Vol. 52",Bizarrap;Quevedo,99
spotify:track:5ww2BF9slyYgNOk37BlC4u,La Bachata,Manuel Turizo,98
spotify:track:4uUG5RXrOk84mYEfFvj3cK,I'm Good (Blue),David Guetta;Bebe Rexha,98
spotify:track:1IHWl5LamUGEuP4ozKQSXZ,Tití Me Preguntó,Bad Bunny,97
spotify:track:6Sq7ltF9Qa7SNFBsV5Cogx,Me Porto Bonito,Bad Bunny;Chencho Corleone,97
spotify:track:5Eax0qFko2dh7Rl2lYs3bx,Efecto,Bad Bunny,96
spotify:track:5IgjP7X4th6nMNDh4akUHb,Under The Influence,Chris Brown,96
spotify:track:4h9wh7iOZ0GGn8QVp4RAOB,I Ain't Worried,OneRepublic,96


These are simply the highest-popularity tracks in the catalog, as expected for a cold playlist.

## Case 2: Warm playlist, mix of warm and cold tracks

Take a real playlist from our processed data with enough tracks to count as
"warm", and route recommendations for it. Tracks the model has seen often
get scored collaboratively; rare tracks fall back to content similarity
against this playlist's own tracks.

In [5]:
playlist_sizes = interactions_df.groupby('pid').size()
warm_pid = playlist_sizes.index[0]
warm_playlist_tracks = interactions_df[interactions_df['pid'] == warm_pid]['track_uri'].tolist()
print(f"Playlist {warm_pid}: {interactions_df[interactions_df['pid']==warm_pid]['playlist_name'].iloc[0]}")
print(f"{len(warm_playlist_tracks)} tracks")

playlist_track_indices = [
    candidate_df.loc[candidate_df['track_uri'] == uri, 'local_idx'].iloc[0]
    for uri in warm_playlist_tracks if uri in local_idx_map
]

recs_warm = router.route_recommendations(
    playlist_idx=warm_pid,
    playlist_track_count=len(warm_playlist_tracks),
    candidate_tracks=candidate_df.assign(track_idx=candidate_df['local_idx']),
    hybrid_model=hybrid_model,
    content_similarity_matrix=sim_matrix_full,
    playlist_track_indices=playlist_track_indices,
)
candidate_df.set_index('track_uri').loc[recs_warm, ['track_name', 'artist_name', 'popularity']]

Playlist 0: Acoustic Mix 0
332 tracks


,track_name,artist_name,popularity
track_uri,,,
spotify:track:0kKuVSnv97w8dZ0yjde1KU,No compasso do criador - Ao vivo,Katinguelê,39
spotify:track:1bngrz3hq1AIdAD5wmzHTr,Minha Fama de Mau - Ao Vivo,Erasmo Carlos,36
spotify:track:43GJmTLEE0itzVPFmCKmmS,Sentado à Beira do Caminho - Ao Vivo,Erasmo Carlos,39
spotify:track:2uJ8FAeLomeAqLejkIQQu4,Múmias,Biquini Cavadão,39
spotify:track:02zFCfzkqFwNaZ4f0idBbi,Me Apaixonei Pela Pessoa Errada - Ao Vivo,Exaltasamba,39
spotify:track:4YeOF5O52bguNs8w2RK30Y,Vento Ventania: Roostie And Boopsie: You Are M...,Biquini Cavadão,39
spotify:track:4FCU1TwsmITjvoVO26Lyae,Ee Hasiru Siriyali,Sangeetha Katti,34
spotify:track:0o60e52PdVZ70e6YHWibSE,How Bad Do You Want It (Oh Yeah),Sevyn Streeter,38
spotify:track:1Gp3235UGtWR50HwDFDjyu,Heart Of Pop (Percayalah),Closehead,38


Note: because notebook 02's crossover step (added to help the
collaborative model generalize - see notebook 05) spreads popular tracks
across many playlists, most tracks here end up with enough interactions to
count as "warm" under the default `warm_track_threshold=10`, so this
recommendation list is dominated by the collaborative-filtering path. The
next section forces the content-based cold-start path explicitly so we can
inspect it in isolation.

## Case 3: Track cold-start, isolated

Use a much higher `warm_track_threshold` so every candidate is treated as track-cold-start, isolating the content-similarity fallback path.

In [6]:
cold_track_router = SpotifyColdStartRouter(warm_playlist_threshold=5, warm_track_threshold=100000)

recs_content_only = cold_track_router.route_recommendations(
    playlist_idx=warm_pid,
    playlist_track_count=len(warm_playlist_tracks),
    candidate_tracks=candidate_df.assign(track_idx=candidate_df['local_idx']),
    hybrid_model=hybrid_model,
    content_similarity_matrix=sim_matrix_full,
    playlist_track_indices=playlist_track_indices,
)
candidate_df.set_index('track_uri').loc[recs_content_only, ['track_name', 'artist_name', 'popularity']]

,track_name,artist_name,popularity
track_uri,,,
spotify:track:5dyRLxyjgy5orlNd9hNCt0,Tu És o Centro (Ao Vivo),Frei Gilson,49
spotify:track:0HL3y3bsKSARBSP0qaImwb,Carried Me With You,Brandi Carlile,62
spotify:track:42tIFNfGtvPewIfkmwXlAL,Onze e Pouquinho - Ao Vivo,Dilsinho,50
spotify:track:3EFljr9vPDiP3Q09CkT1Y3,Algo Maior - Ao Vivo,Luma Elpidio;Bruna Branco;LE MUSIC;Nathália Bl...,42
spotify:track:0ViZGhhFZtaTA6O9H94ki6,Cachorro Caramelo - Ao Vivo,Clayton & Romário,52
spotify:track:6uGDB28Q5oyYCgP6RigmIj,Sacramento da Comunhão (Ao Vivo),Nelsinho Corrêa;Ana Lúcia,52
spotify:track:4kaLkst9lT3sQ5Crgxs3ic,Pani Da Rang - Male Vocals,Ayushmann Khurrana,68
spotify:track:6Cnwfw1YmMQQMb4femS10J,Fã De Erro - Ao Vivo,Guga Nandes,49
spotify:track:6pnkWwsWrwWY00Xpw3pc3b,novo - Ao Vivo,Leonardo Gonçalves,41


## Before/after: why the `playlist_track_indices` fix matters

With the original bug, content-based scoring for cold tracks ignored the
playlist entirely and just returned each candidate's average similarity to
the whole catalog - so the ranking of cold tracks would be *identical* no
matter which playlist you were recommending for. Let's confirm the fix
actually makes the ranking playlist-dependent.

In [7]:
def content_rank_for_playlist(track_indices):
    return router._compute_content_score(
        track_idx=probe_local_idx,
        playlist_track_indices=track_indices,
        sim_matrix=sim_matrix_full,
    )

# probe with a candidate track that isn't already in either playlist
probe_uri = candidate_df.loc[~candidate_df['track_uri'].isin(warm_playlist_tracks), 'track_uri'].iloc[0]
probe_local_idx = candidate_df.loc[candidate_df['track_uri'] == probe_uri, 'local_idx'].iloc[0]
print("Probe track:", candidate_df.loc[candidate_df['track_uri'] == probe_uri, 'track_name'].iloc[0])

# a genre-distinct second playlist, picked from a different point in the sorted pid list
other_candidates = playlist_sizes.index[playlist_sizes.index != warm_pid]
other_pid = other_candidates[len(other_candidates) // 2]
other_playlist_tracks = interactions_df[interactions_df['pid'] == other_pid]['track_uri'].tolist()
print(f"Comparison playlist {other_pid}: {interactions_df[interactions_df['pid']==other_pid]['playlist_name'].iloc[0]} ({len(other_playlist_tracks)} tracks)")

other_track_indices = [
    candidate_df.loc[candidate_df['track_uri'] == uri, 'local_idx'].iloc[0]
    for uri in other_playlist_tracks if uri in local_idx_map
]

score_vs_playlist_a = content_rank_for_playlist(playlist_track_indices)
score_vs_playlist_b = content_rank_for_playlist(other_track_indices)
score_vs_whole_catalog = router._compute_content_score(probe_local_idx, [], sim_matrix_full)

print(f"\nScore vs playlist {warm_pid} tracks: {score_vs_playlist_a:.4f}")
print(f"Score vs playlist {other_pid} tracks: {score_vs_playlist_b:.4f}")
print(f"Score vs whole catalog (old buggy behavior): {score_vs_whole_catalog:.4f}")

Probe track: Unholy (feat. Kim Petras)
Comparison playlist 337: Indie Mix 337 (320 tracks)



Score vs playlist 0 tracks: 0.0043
Score vs playlist 337 tracks: 0.0034
Score vs whole catalog (old buggy behavior): 0.0044


The two playlist-specific scores differ from each other and from the old whole-catalog average, confirming the fix makes content-based cold-start scoring actually playlist-aware.